<a href="https://colab.research.google.com/github/p-tech/BDAI-Files/blob/main/API/tefl_api_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with the TfL Open API

## Workbook introduction

This workbook introduces the **Transport for London (TfL) Unified API** and shows how to retrieve, inspect, transform, and use live transport data in Python.

TfL publishes a wide range of open datasets and real-time information through its API, covering services including the Underground, buses, Overground, Elizabeth line, trams, river services, Santander Cycles, road disruption data, and journey planning. The API returns information in **JSON** format, which we will request over the web and analyse using Python tools such as `requests` and `pandas`.

The goal is not only to make individual API calls, but to develop a repeatable workflow for working with web APIs:

1. Identify the endpoint and the data it provides.
2. Send a request with the appropriate parameters and credentials.
3. Check that the response was successful and understand its JSON structure.
4. Extract the fields needed for analysis.
5. Convert the results into a structured table or visualisation.
6. Handle missing values, empty results, and request errors safely.

## Learning objectives

By the end of this workbook, you should be able to:

- explain what an API is and why it is useful for accessing live public data;
- make HTTP `GET` requests to the TfL Unified API from Google Colab;
- use query parameters, including an application ID and API key where required;
- inspect and navigate JSON responses;
- create pandas DataFrames from API data;
- retrieve examples of live TfL information, such as service status, arrivals, or cycle-hire availability; and
- adapt an example request to answer a new transport-data question.

## The TfL Unified API

An **Application Programming Interface (API)** is a defined way for one program to request data or services from another. In this workbook, Python is the client: it sends a request to a TfL API endpoint and receives a response containing the requested data.

TfL's base API address is:

```text
https://api.tfl.gov.uk
```

An **endpoint** is the part of the address that identifies a particular service or dataset. For example, an endpoint can return the current status of one or more transport lines. Requests can also include **parameters** to specify exactly what data is needed.

Many TfL endpoints can be explored without authentication. For regular, automated, or higher-volume use, requests should include a TfL `app_id` and `app_key`. These credentials identify the application making the request and help TfL manage fair use of the service. They should not be hard-coded into a shared notebook or committed to a public repository. In Colab, we will store them as secrets and load them only when needed.

## How this workbook will work

Each practical section will follow the same pattern:

- **Question** — define the transport question we want to answer.
- **API request** — construct and run the URL or Python request.
- **Inspect** — examine the response and identify useful fields.
- **Transform** — organise the data into a DataFrame where appropriate.
- **Interpret** — explain what the output tells us and note any limitations.

Live API data changes continuously. Running the same cell at different times can therefore return different arrival predictions, service statuses, availability counts, or disruption details. This is expected and is one of the benefits—and analytical considerations—of using a real-time API.

## Before we begin

This workbook assumes basic familiarity with Python variables, lists, dictionaries, and pandas DataFrames. No previous API experience is required.

As we work, pay attention to three practical questions:

- What does this endpoint provide?
- What is the structure of the returned JSON?
- How can the response be turned into information that answers a useful question?

We will begin by importing the Python libraries needed to send requests and work with returned data.

# 1. Set up the notebook

## What we are doing

Before making an API call, we need to prepare the notebook. We will:

1. import Python libraries for making web requests and working with tabular data;
2. store the TfL API base address in one reusable variable; and
3. make a simple request to check that the API is available.

For this classroom activity, no API registration, `app_id`, or `app_key` is required. We will use publicly accessible TfL API endpoints and keep the focus on making and analysing requests.

## 1.1 Import the libraries

Run the following code cell. `requests` sends HTTP requests to the API, while `pandas` helps us organise returned data into tables.

Google Colab already includes both packages, so no installation should be needed.

In [ ]:
import pandas as pd
import requests

# Display all columns when viewing a pandas DataFrame.
pd.set_option("display.max_columns", None)

## 1.2 Define the base URL

All TfL Unified API endpoints begin with the same base address. Store it once so that later requests can add only the endpoint they need.

For example, the URL for the line-status endpoint will later be formed by combining `BASE_URL` with `/Line/Mode/tube/Status`.

In [ ]:
BASE_URL = "https://api.tfl.gov.uk"

## 1.3 Check that the API is available

TfL provides a `Meta` endpoint that returns general information about the API. This is a useful first request because it is publicly accessible and does not require us to choose a particular transport service.

In [ ]:
response = requests.get(f"{BASE_URL}/Meta", timeout=30)

print("Status code:", response.status_code)
response.raise_for_status()

meta = response.json()
print("Response type:", type(meta).__name__)

A status code of `200` means the request was successful. The `response.raise_for_status()` line stops the cell and reports an error if the API returns an unsuccessful response, such as `404` (endpoint not found) or `500` (server error).

## Checkpoint

At this point, the notebook has a reusable API base URL and the tools needed to make public requests to TfL. The next section will make a first meaningful call: retrieving the current status of London Underground lines.
nebulaONE logo

In [ ]:
import requests
import pandas as pd

# TfL API endpoint for Tube line status
url = "https://api.tfl.gov.uk/Line/Mode/tube/Status"

# Send request
response = requests.get(url)

# Check if request worked
if response.status_code == 200:
    data = response.json()

    # Extract useful information
    results = []

    for line in data:
        line_name = line["name"]
        status = line["lineStatuses"][0]["statusSeverityDescription"]
        reason = line["lineStatuses"][0].get("reason", "No details provided")

        results.append({
            "Line": line_name,
            "Status": status,
            "Details": reason
        })

    # Convert to DataFrame for clean display
    df = pd.DataFrame(results)
    print(df)

else:
    print("Error:", response.status_code)


In [ ]:
import pandas as pd

# Create a DataFrame
df1 = pd.DataFrame(results)

# For even richer display in Colab, you can just type the DataFrame name at the end of a cell
# This also works with `display(sample_df)`
df1